# Toy: Grade a Student Slide Deck with LandingAI ADE

Minimal prototype of using LandingAI's Agentic Document Extraction (ADE) as a primitive for the `grade-deck` / `review-deck` skills.

**Goal.** Take a student PDF deck and, in one API call per capability:

1. **Parse** the deck into per-slide markdown + per-chunk bounding boxes + chunk types (`text`, `table`, `figure`, `marginalia`).
2. **Extract** rubric-relevant fields (action titles, exec summary, sources, densities) into a typed Pydantic schema with visual grounding.
3. **Score** a subset of the Deck Quality Rubric deterministically from the extracted structure.

This is a toy: we wire the plumbing end-to-end on one deck and look at the outputs. It is not the full rubric.

**Prereq.** Set `VISION_AGENT_API_KEY` in `.env` or your shell. Get a free key at [va.landing.ai](https://va.landing.ai).

## 1. Setup

In [ ]:
%pip install -q landingai-ade pydantic pymupdf pillow python-dotenv pandas

In [ ]:
import os
import json
from pathlib import Path
from enum import Enum
from typing import List, Optional

import pandas as pd
import pymupdf
from dotenv import load_dotenv
from PIL import Image, ImageDraw
from IPython.display import display, HTML, Markdown

from pydantic import BaseModel, Field
from landingai_ade import LandingAIADE
from landingai_ade.types import ParseResponse, ExtractResponse
from landingai_ade.lib import pydantic_to_json_schema

# Load VISION_AGENT_API_KEY from .env.local (falls back to .env)
load_dotenv(dotenv_path=".env.local", override=True)
load_dotenv(override=True)
assert os.getenv("VISION_AGENT_API_KEY"), "Set VISION_AGENT_API_KEY in .env.local"

client = LandingAIADE()  # SDK reads VISION_AGENT_API_KEY from env automatically
print("Authenticated ADE client ready.")

In [ ]:
# Point this at any student deck PDF.
DECK_PATH = Path("sample.pdf")
assert DECK_PATH.exists(), f"Deck not found: {DECK_PATH.resolve()}"
print(f"Grading: {DECK_PATH.name} ({DECK_PATH.stat().st_size/1024:.0f} KB)")

## 2. Parse the deck, split by page

`split="page"` gives one markdown blob and one chunk list per slide. Each chunk has a type, a bounding box, and a grounding ID we can use for citations.

In [ ]:
print("Calling Parse API...")
parse_result: ParseResponse = client.parse(
    document=DECK_PATH,
    model="dpt-2-latest",
    split="page",
)
print(f"Slides parsed: {len(parse_result.splits)}")
print(f"Total chunks:  {len(parse_result.chunks)}")
print(f"Duration (ms): {parse_result.metadata.duration_ms}")

In [ ]:
# Per-slide summary: chunk counts by type, word count.
def slide_summary(parse_result) -> pd.DataFrame:
    rows = []
    for i, split in enumerate(parse_result.splits, start=1):
        md = split.markdown or ""
        chunks = [c for c in parse_result.chunks if c.grounding.page == i - 1]
        type_counts = {}
        for c in chunks:
            type_counts[c.type] = type_counts.get(c.type, 0) + 1
        rows.append({
            "slide": i,
            "words": len(md.split()),
            "chunks": len(chunks),
            "figures": type_counts.get("figure", 0),
            "tables": type_counts.get("table", 0),
            "text": type_counts.get("text", 0),
            "marginalia": type_counts.get("marginalia", 0),
        })
    return pd.DataFrame(rows)

summary_df = slide_summary(parse_result)
display(summary_df)
print(f"Avg words/slide: {summary_df.words.mean():.0f}  |  Max: {summary_df.words.max()}")

### 2a. Peek at one slide's markdown

In [ ]:
PEEK_SLIDE = 2  # 1-indexed
display(Markdown(f"**Slide {PEEK_SLIDE} markdown:**"))
print(parse_result.splits[PEEK_SLIDE - 1].markdown[:1500])

### 2b. Visualize chunks on one slide

Render the PDF page at 150 DPI and draw each chunk's bounding box. Color by type. This is the image the `review-deck` skill's design checks would operate on.

In [ ]:
TYPE_COLORS = {
    "text": "#2563eb",
    "table": "#16a34a",
    "figure": "#dc2626",
    "marginalia": "#9333ea",
    "logo": "#ea580c",
}

def render_slide_with_boxes(parse_result, pdf_path: Path, slide_num: int, dpi: int = 150) -> Image.Image:
    doc = pymupdf.open(pdf_path)
    page = doc[slide_num - 1]
    pix = page.get_pixmap(dpi=dpi)
    img = Image.frombytes("RGB", (pix.width, pix.height), pix.samples).copy()
    draw = ImageDraw.Draw(img)
    W, H = pix.width, pix.height
    for c in parse_result.chunks:
        if c.grounding.page != slide_num - 1:
            continue
        box = c.grounding.box  # normalized [l, t, r, b]
        x0, y0, x1, y1 = box.l * W, box.t * H, box.r * W, box.b * H
        color = TYPE_COLORS.get(c.type, "#6b7280")
        draw.rectangle([x0, y0, x1, y1], outline=color, width=3)
        draw.text((x0 + 4, y0 + 4), c.type, fill=color)
    doc.close()
    return img

render_slide_with_boxes(parse_result, DECK_PATH, PEEK_SLIDE)

## 3. Extract rubric-relevant fields with a Pydantic schema

The `grade-deck` skill cares about four dimensions: Storyline, Insight, Evidence, Design. For a toy we'll extract just the few fields most machine-extractable from the deck itself:

- The action title of each slide
- Whether an executive summary slide exists, and its core recommendation
- Whether a sources / appendix slide exists
- Target company and student name (for the header of the student-facing grade file)

In [ ]:
class SlideTitle(BaseModel):
    slide_number: int = Field(description="1-indexed slide number")
    title: str = Field(description="The action title or slide headline. If the slide has no title, return an empty string.")
    is_action_title: bool = Field(description="True if the title states a conclusion (e.g., 'X drives 40% of growth'). False if it's a topic label (e.g., 'Market Overview').")

class DeckExtraction(BaseModel):
    target_company: str = Field(description="The public company the deck is proposing to. Leave blank if unclear.")
    student_name: str = Field(description="The student author's name from the title slide. Leave blank if not shown.")
    slide_titles: List[SlideTitle] = Field(description="One entry per slide, in order.")
    has_executive_summary: bool = Field(description="True if the deck contains a dedicated executive summary slide near the front.")
    executive_summary_recommendation: Optional[str] = Field(description="The one-sentence recommended action from the exec summary, if present.")
    has_sources_slide: bool = Field(description="True if the deck has a sources, references, or bibliography slide in the appendix.")
    has_course_artifacts_in_titles: bool = Field(description="True if any slide title contains course artifacts like 'P1', 'P2', 'STRAT 325', or similar assignment labels.")

schema_json = json.dumps(pydantic_to_json_schema(DeckExtraction))
print("Schema built.")

In [ ]:
print("Calling Extract API...")
extract_result: ExtractResponse = client.extract(
    schema=schema_json,
    markdown=parse_result.markdown,
    model="extract-latest",
)
extracted = DeckExtraction.model_validate(extract_result.extraction)
print(f"Target company: {extracted.target_company}")
print(f"Student:        {extracted.student_name}")
print(f"Exec summary:   {extracted.has_executive_summary}  |  Sources slide: {extracted.has_sources_slide}")
print(f"Course artifacts in titles: {extracted.has_course_artifacts_in_titles}")
print(f"Recommendation: {extracted.executive_summary_recommendation}")

In [ ]:
titles_df = pd.DataFrame([t.model_dump() for t in extracted.slide_titles])
display(titles_df)

## 4. Toy scoring: fire a few patterns deterministically

Real `grade-deck` has dozens of patterns. Here we fire three as a smoke test:

| Pattern | Rule | Dimension |
|---|---|---|
| Client-Readiness Gate | `has_course_artifacts_in_titles` OR NOT `has_executive_summary` | Storyline cap |
| Topic-labels-not-conclusions (S-weak) | < 50% of titles are action titles | Storyline |
| Density warning (D-C05a) | avg words/slide > 180 OR max > 250 | Design |

In [ ]:
findings = []

# Client-Readiness Gate
if extracted.has_course_artifacts_in_titles:
    findings.append({"pattern": "Client-Readiness Gate", "dimension": "Storyline", "severity": "CAP=4", "note": "Course artifacts (P1/P2/STRAT 325) appear in titles."})
if not extracted.has_executive_summary:
    findings.append({"pattern": "Client-Readiness Gate", "dimension": "Storyline", "severity": "CAP=4", "note": "No executive summary slide detected."})

# Topic labels
action_ratio = titles_df["is_action_title"].mean() if len(titles_df) else 0
if action_ratio < 0.5:
    findings.append({"pattern": "Titles are topic labels", "dimension": "Storyline", "severity": "T2", "note": f"Only {action_ratio:.0%} of titles state a conclusion."})

# Density
avg_w = summary_df.words.mean()
max_w = summary_df.words.max()
if avg_w > 180 or max_w > 250:
    findings.append({"pattern": "D-C05a density", "dimension": "Design", "severity": "T2", "note": f"Avg {avg_w:.0f} w/slide, max {max_w}. Reader being asked to read, not scan."})

# Sources
if not extracted.has_sources_slide:
    findings.append({"pattern": "Missing sources appendix", "dimension": "Evidence", "severity": "T2", "note": "No sources/bibliography slide detected in appendix."})

pd.DataFrame(findings) if findings else Markdown("_No patterns fired._")

## 5. Student-facing header (demo)

Show how the extracted structure plugs into the `grade-deck` output format.

In [ ]:
core_slide_count = len(titles_df) - 2  # rough: exclude title + one divider
output = f"""# Deck Quality Assessment

**Student:** {extracted.student_name or '(unknown)'}
**Company:** {extracted.target_company or '(unknown)'}
**Core slides:** {core_slide_count}

## Action Titles
"""
for t in extracted.slide_titles:
    mark = "" if t.is_action_title else "  (topic label)"
    output += f"- Slide {t.slide_number}: \"{t.title}\"{mark}\n"

output += f"""
## Findings (toy subset)
"""
for f in findings:
    output += f"- **{f['pattern']}** ({f['dimension']}, {f['severity']}): {f['note']}\n"

display(Markdown(output))

## What this demonstrates for the skills

- **One Parse call** replaces the current `Read(pages=...)` batched-image approach and gives us *structured* output: per-slide markdown + per-chunk bounding boxes + chunk types.
- **One Extract call** with a Pydantic schema pulls exactly the rubric-relevant fields we want. The schema IS the rubric contract.
- **Deterministic pattern firing** runs over the extracted structure, not over vibes. Rules are testable in isolation.
- **Visual grounding** means each finding can cite a specific bounding box on a specific slide for the student-facing report.

**Next moves to productionize:**
1. Expand the Pydantic schema to cover all pattern inputs (sources-per-claim, chart-takeaway-captions, exec-summary content, etc.).
2. Move pattern rules out of `patterns.md` prose and into Python so they're unit-testable.
3. Keep the vision-model pass for design patterns that can't be extracted as text (squint test, visual variety, color consistency). Route cropped chunk images to Claude/GPT-4o instead of sending the full slide.
4. Wrap as a CLI that the skill's instructions can invoke: `python grade_deck.py path/to/deck.pdf > grades/student.md`.